# Filtre Modülü Testi (`filter_dataset`)

`src/analysis/filters.py` içindeki `filter_dataset` fonksiyonunu farklı parametrelerle test eder.

- **Girdi:** `data/processed/PTLD00AUS_2022001_30S.parquet`
- **Çıktı dosyası yok** — sadece filtrelerin beklenen şekilde çalışıp çalışmadığı kontrol edilir.

In [6]:
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.analysis.filters import filter_dataset

PARQUET_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "PTLD00AUS_2022001_30S.parquet")

In [7]:
df = pd.read_parquet(PARQUET_PATH)

print(f"Dosya: {PARQUET_PATH}")
print(f"Satır sayısı: {len(df):,}")
print(f"Sütunlar: {list(df.columns)}")
print(f"Epoch aralığı : {df['epoch'].min()} → {df['epoch'].max()}")
print(f"Elevation     : {df['elevation'].min():.2f}° – {df['elevation'].max():.2f}°")
print(f"Azimuth       : {df['azimuth'].min():.2f}° – {df['azimuth'].max():.2f}°")
print(f"Wavelength    : {df['wavelength'].min():.4f} – {df['wavelength'].max():.4f} m")
print(f"Constellation : {sorted(df['constellation'].unique())}")
print(f"obsType sayısı: {df['obsType'].nunique()}  |  satID sayısı: {df['satID'].nunique()}")

Dosya: c:\vscode_workspace\PyReflection\data\processed\PTLD00AUS_2022001_30S.parquet
Satır sayısı: 422,923
Sütunlar: ['epoch', 'constellation', 'satID', 'arcNo', 'arcType', 'obsType', 'wavelength', 'obsValue', 'elevation', 'azimuth']
Epoch aralığı : 2022-01-01 00:00:00 → 2022-01-01 23:59:30
Elevation     : -0.12° – 89.36°
Azimuth       : 0.00° – 360.00°
Wavelength    : 0.1867 – 0.2548 m
Constellation : ['C', 'E', 'G', 'J', 'R']
obsType sayısı: 16  |  satID sayısı: 112


In [8]:
def _in_any_range(values, ranges):
    mask = np.zeros(len(values), dtype=bool)
    for lo, hi in ranges:
        mask |= (values >= lo) & (values <= hi)
    return mask


def _in_azim_ranges(values, ranges):
    mask = np.zeros(len(values), dtype=bool)
    for lo, hi in ranges:
        if lo > hi:
            mask |= (values >= lo) | (values <= hi)
        else:
            mask |= (values >= lo) & (values <= hi)
    return mask


def validate_filtered(df_in, df_out, expect_empty=False, **kwargs):
    """Apply post-filter checks and return a list of (check_name, passed, detail)."""
    checks = []

    if expect_empty:
        checks.append(("expect_empty", df_out.empty, f"satır sayısı={len(df_out)}"))
    elif df_out.empty and not df_in.empty and any(
        v is not None and (not isinstance(v, list) or len(v) > 0)
        for v in kwargs.values()
    ):
        checks.append(("non_empty_when_filters_applied", False, "Filtre sonrası veri boş"))
    else:
        checks.append(("non_empty_when_filters_applied", True, f"{len(df_out):,} satır kaldı"))

    if df_out.empty:
        return checks

    elev_ranges = kwargs.get("elev_ranges")
    if elev_ranges:
        ok = _in_any_range(df_out["elevation"].to_numpy(), elev_ranges).all()
        checks.append(("elevation", ok, f"aralıklar={elev_ranges}"))

    azim_ranges = kwargs.get("azim_ranges")
    if azim_ranges:
        ok = _in_azim_ranges(df_out["azimuth"].to_numpy(), azim_ranges).all()
        checks.append(("azimuth", ok, f"aralıklar={azim_ranges}"))

    wavelength_ranges = kwargs.get("wavelength_ranges")
    if wavelength_ranges and "wavelength" in df_out.columns:
        ok = _in_any_range(df_out["wavelength"].to_numpy(), wavelength_ranges).all()
        checks.append(("wavelength", ok, f"aralıklar={wavelength_ranges}"))

    obs_types_include = kwargs.get("obs_types_include")
    if obs_types_include:
        ok = df_out["obsType"].isin(obs_types_include).all()
        checks.append(("obs_types_include", ok, f"izin verilen={obs_types_include}"))

    obs_types_exclude = kwargs.get("obs_types_exclude")
    if obs_types_exclude:
        ok = ~df_out["obsType"].isin(obs_types_exclude).any()
        checks.append(("obs_types_exclude", ok, f"hariç tutulan={obs_types_exclude}"))

    constellations = kwargs.get("constellations")
    if constellations:
        ok = df_out["constellation"].isin(constellations).all()
        checks.append(("constellations", ok, f"izin verilen={constellations}"))

    sat_ids = kwargs.get("sat_ids")
    if sat_ids:
        ok = df_out["satID"].isin(sat_ids).all()
        checks.append(("sat_ids", ok, f"izin verilen={sat_ids}"))

    epoch_start = kwargs.get("epoch_start")
    if epoch_start:
        ok = (df_out["epoch"] >= pd.to_datetime(epoch_start)).all()
        checks.append(("epoch_start", ok, f">= {epoch_start}"))

    epoch_end = kwargs.get("epoch_end")
    if epoch_end:
        ok = (df_out["epoch"] <= pd.to_datetime(epoch_end)).all()
        checks.append(("epoch_end", ok, f"<= {epoch_end}"))

    return checks


def run_filter_test(name, df_in, expected_subset_of=None, expect_empty=False, **filter_kwargs):
    """Run one filter scenario and print a compact PASS/FAIL report."""
    filtered = filter_dataset(df_in, **filter_kwargs)
    checks = validate_filtered(df_in, filtered, expect_empty=expect_empty, **filter_kwargs)

    if expected_subset_of is not None:
        ok = len(filtered) == expected_subset_of.sum()
        checks.append(("expected_row_count", ok, f"beklenen={expected_subset_of.sum():,}, gerçek={len(filtered):,}"))

    all_ok = all(passed for _, passed, _ in checks)

    print(f"\n{'=' * 70}")
    print(f"TEST: {name}")
    print(f"Parametreler: {filter_kwargs or '(filtre yok)'}")
    print(f"Girdi: {len(df_in):,} satır  →  Çıktı: {len(filtered):,} satır  ({len(filtered)/len(df_in)*100:.1f}%)")
    print(f"SONUÇ: {'✓ PASS' if all_ok else '✗ FAIL'}")
    for check_name, passed, detail in checks:
        status = "OK" if passed else "HATA"
        print(f"  [{status}] {check_name}: {detail}")

    return {
        "name": name,
        "input_rows": len(df_in),
        "output_rows": len(filtered),
        "passed": all_ok,
        "checks": checks,
        "filtered_df": filtered,
    }

## Filtre Senaryoları

Her senaryo için:
1. `filter_dataset` çalıştırılır
2. Kalan satırların filtre kriterlerine uyup uymadığı doğrulanır
3. Bazı testlerde beklenen satır sayısı manuel maske ile karşılaştırılır

In [9]:
results = []

# 1) Filtre yok — tüm satırlar korunmalı
results.append(run_filter_test(
    "No filter (baseline)",
    df,
    expected_subset_of=pd.Series(True, index=df.index),
))

# 2) Elevation — tek aralık
elev_mask = (df["elevation"] >= 10) & (df["elevation"] <= 30)
results.append(run_filter_test(
    "Elevation single range",
    df,
    expected_subset_of=elev_mask,
    elev_ranges=[(10, 30)],
))

# 3) Elevation — birden çok aralık (OR)
elev_mask_multi = ((df["elevation"] >= 0) & (df["elevation"] <= 5)) | ((df["elevation"] >= 80) & (df["elevation"] <= 90))
results.append(run_filter_test(
    "Elevation multiple ranges (OR)",
    df,
    expected_subset_of=elev_mask_multi,
    elev_ranges=[(0, 5), (80, 90)],
))

# 4) Azimuth — normal aralık
azim_mask = (df["azimuth"] >= 90) & (df["azimuth"] <= 180)
results.append(run_filter_test(
    "Azimuth normal range",
    df,
    expected_subset_of=azim_mask,
    azim_ranges=[(90, 180)],
))

# 5) Azimuth — kuzey sarmalaması (350° → 10°)
azim_wrap_mask = (df["azimuth"] >= 350) | (df["azimuth"] <= 10)
results.append(run_filter_test(
    "Azimuth north wrap-around",
    df,
    expected_subset_of=azim_wrap_mask,
    azim_ranges=[(350, 10)],
))

# 6) Wavelength
wl_mask = (df["wavelength"] >= 0.19) & (df["wavelength"] <= 0.24)
results.append(run_filter_test(
    "Wavelength range",
    df,
    expected_subset_of=wl_mask,
    wavelength_ranges=[(0.19, 0.24)],
))

# 7) obsType — dahil et
obs_include_mask = df["obsType"].isin(["S1C", "S1X"])
results.append(run_filter_test(
    "obsType include",
    df,
    expected_subset_of=obs_include_mask,
    obs_types_include=["S1C", "S1X"],
))

# 8) obsType — hariç tut
obs_exclude_mask = ~df["obsType"].isin(["S2W", "S2P"])
results.append(run_filter_test(
    "obsType exclude",
    df,
    expected_subset_of=obs_exclude_mask,
    obs_types_exclude=["S2W", "S2P"],
))

# 9) Constellation — yalnızca GPS ve Galileo
const_mask = df["constellation"].isin(["G", "E"])
results.append(run_filter_test(
    "Constellation filter",
    df,
    expected_subset_of=const_mask,
    constellations=["G", "E"],
))

# 10) satID — belirli uydular
sample_sats = ["G01", "G02", "E01", "E02"]
sat_mask = df["satID"].isin(sample_sats)
results.append(run_filter_test(
    "satID filter",
    df,
    expected_subset_of=sat_mask,
    sat_ids=sample_sats,
))

# 11) Epoch — günün ilk 2 saati
epoch_mask = (df["epoch"] >= "2022-01-01 00:00:00") & (df["epoch"] <= "2022-01-01 02:00:00")
results.append(run_filter_test(
    "Epoch time window",
    df,
    expected_subset_of=epoch_mask,
    epoch_start="2022-01-01 00:00:00",
    epoch_end="2022-01-01 02:00:00",
))

# 12) Kombine filtre — main.py ile aynı örnek
combo_mask = (df["elevation"] >= 5) & (df["elevation"] <= 25)
combo_mask &= wl_mask
combo_mask &= df["obsType"].isin(["S1C"])
combo_mask &= ~df["obsType"].isin(["S2W"])
results.append(run_filter_test(
    "Combined filters (main.py example)",
    df,
    expected_subset_of=combo_mask,
    elev_ranges=[(5, 25)],
    wavelength_ranges=[(0.19, 0.24)],
    obs_types_include=["S1C"],
    obs_types_exclude=["S2W"],
))

# 13) Çok kısıtlayıcı filtre — boş sonuç beklenir
results.append(run_filter_test(
    "Over-restrictive filter (expect empty)",
    df,
    expect_empty=True,
    elev_ranges=[(89, 90)],
    azim_ranges=[(0, 1)],
    constellations=["J"],
    obs_types_include=["S1C"],
    sat_ids=["G99"],
))


TEST: No filter (baseline)
Parametreler: (filtre yok)
Girdi: 422,923 satır  →  Çıktı: 422,923 satır  (100.0%)
SONUÇ: ✓ PASS
  [OK] non_empty_when_filters_applied: 422,923 satır kaldı
  [OK] expected_row_count: beklenen=422,923, gerçek=422,923

TEST: Elevation single range
Parametreler: {'elev_ranges': [(10, 30)]}
Girdi: 422,923 satır  →  Çıktı: 129,580 satır  (30.6%)
SONUÇ: ✓ PASS
  [OK] non_empty_when_filters_applied: 129,580 satır kaldı
  [OK] elevation: aralıklar=[(10, 30)]
  [OK] expected_row_count: beklenen=129,580, gerçek=129,580

TEST: Elevation multiple ranges (OR)
Parametreler: {'elev_ranges': [(0, 5), (80, 90)]}
Girdi: 422,923 satır  →  Çıktı: 37,395 satır  (8.8%)
SONUÇ: ✓ PASS
  [OK] non_empty_when_filters_applied: 37,395 satır kaldı
  [OK] elevation: aralıklar=[(0, 5), (80, 90)]
  [OK] expected_row_count: beklenen=37,395, gerçek=37,395

TEST: Azimuth normal range
Parametreler: {'azim_ranges': [(90, 180)]}
Girdi: 422,923 satır  →  Çıktı: 99,866 satır  (23.6%)
SONUÇ: ✓ PASS


In [10]:
summary = pd.DataFrame([
    {
        "test": r["name"],
        "input_rows": r["input_rows"],
        "output_rows": r["output_rows"],
        "pass": r["passed"],
    }
    for r in results
])

passed_count = summary["pass"].sum()
total_count = len(summary)

print(f"\nÖZET: {passed_count}/{total_count} test geçti")
display(summary)

if passed_count == total_count:
    print("\nTüm filtreler beklenen şekilde çalışıyor.")
else:
    failed = summary.loc[~summary["pass"], "test"].tolist()
    print(f"\nBaşarısız testler: {failed}")


ÖZET: 13/13 test geçti


,test,input_rows,output_rows,pass
0,No filter (baseline),422923,422923,True
1,Elevation single range,422923,129580,True
2,Elevation multiple ranges (OR),422923,37395,True
3,Azimuth normal range,422923,99866,True
4,Azimuth north wrap-around,422923,24252,True
5,Wavelength range,422923,218988,True
6,obsType include,422923,118487,True
7,obsType exclude,422923,372686,True
8,Constellation filter,422923,175837,True
9,satID filter,422923,13071,True



Tüm filtreler beklenen şekilde çalışıyor.
